# Model Generalization
Author: Joel Enrique Díaz Villanueva

Organization: Universidad de Monterrey   

Created: 3 March 2026   

---

## Importing the model via MLflow 

In [1]:
from xgboost import XGBClassifier
import pprint

# 1. Initialize an "empty" model object
model = XGBClassifier()

# 2. Load the physical model file from your local path
# This file contains the actual trained decision trees (the "brain")
model_path = "C:\\Users\\USER\\DESKTOP\\PEF\\Terrain-Traversability-Analysis\\xgboost_mlflow_model.json"
model.load_model(model_path)

# 3. Extract hyperparameters from the Python wrapper
# Note: JSON format might show 'None' for some fields as it prioritizes 
# the internal C++ booster structure over Python-specific labels.
params = model.get_params()

print("\n--- MODEL HYPERPARAMETERS (PYTHON WRAPPER) ---")
pprint.pprint(params)

# 4. INTERNAL VALIDATION: Check the actual "Booster" content
# This proves the model is NOT empty despite the 'None' values above.
booster = model.get_booster()
print("\n--- INTERNAL MODEL CHECK ---")
print(f"Total trained trees (n_estimators): {len(booster.get_dump())}")
print(f"Feature names detected: {booster.feature_names}")


--- MODEL HYPERPARAMETERS (PYTHON WRAPPER) ---
{'base_score': [0.21017604, 0.20546962, 0.20630637, 0.15668441, 0.22136354],
 'booster': 'gbtree',
 'callbacks': None,
 'colsample_bylevel': None,
 'colsample_bynode': None,
 'colsample_bytree': None,
 'device': None,
 'early_stopping_rounds': None,
 'enable_categorical': False,
 'eval_metric': None,
 'feature_types': ['float',
                   'float',
                   'float',
                   'float',
                   'float',
                   'int',
                   'int',
                   'int'],
 'feature_weights': None,
 'gamma': None,
 'grow_policy': None,
 'importance_type': None,
 'interaction_constraints': None,
 'learning_rate': None,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': None,
 'max_leaves': None,
 'min_child_weight': None,
 'missing': nan,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_estimators': None,
 'n_jobs': None,
 'n

## Testing the XGBoost model with unseen data 

In [2]:
import polars as pl
import glob
import os
import warnings
warnings.filterwarnings("ignore")

class_name = ['adoquin', 'asfalto', 'concreto', 'grava', 'pasto']
terrain_map = {0: 'Cobblestone', 1: 'Asphalt', 2: 'Concrete', 3: 'Gravel', 4: 'Grass'}

for class_label in class_name:
    print(f"\nClass: {class_label.title()}\n")
    base_path = rf'C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Vectorized_Features_K30_Generalization\{class_label}'
    csv_files = glob.glob(os.path.join(base_path, '**', '*.csv'), recursive=True)
    
    total = []
    
    for csv_file in csv_files:
        print(f"Processing file: {csv_file}")
        
        df = pl.read_csv(csv_file)

        print(f"Loaded Point Cloud with {df.height} points.")

        predictions = model.predict(df)
        
        total.extend(predictions)

        df = df.with_columns([
            pl.Series("terrain_id", predictions),
            pl.Series("terrain_label", [terrain_map[p] for p in predictions])
        ])
        
    if total:
        df_total = pl.DataFrame({
            "terrain_label": [terrain_map[p] for p in total]
        })
        print(f"\n=========================================")
        print(f" Summary for: {class_label.title()}")
        print(f"=========================================")
        print(df_total.group_by("terrain_label").count().sort("count", descending=True))


Class: Adoquin

Processing file: C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Vectorized_Features_K30_Generalization\adoquin\adoquin_test_3_0_features.csv
Loaded Point Cloud with 22492 points.
Processing file: C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Vectorized_Features_K30_Generalization\adoquin\adoquin_test_3_22_features.csv
Loaded Point Cloud with 22492 points.
Processing file: C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Vectorized_Features_K30_Generalization\adoquin\adoquin_test_3_36_features.csv
Loaded Point Cloud with 22505 points.
Processing file: C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Vectorized_Features_K30_Generalization\adoquin\adoquin_test_3_39_features.csv
Loaded Point Cloud with 22491 points.
Processing file: C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Vectorized_Features_K30_Generalization\adoquin\adoquin_test_3_41_features.csv
Loaded Point Cloud with 22492 points.
Processing file: C:\Users\USE

## Plotting the labeled point cloud

In [ ]:
import polars as pl
import plotly.express as px

files = [
    r'C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Vectorized_Features_K30_Generalization\adoquin\adoquin_test_3_0_features.csv',
    r'C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Vectorized_Features_K30_Generalization\asfalto\asfalto_0_features.csv',
    r'C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Vectorized_Features_K30_Generalization\concreto\cemento_ccu_47_features.csv',
    r'C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Vectorized_Features_K30_Generalization\grava\grava_30_features.csv',
    r'C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Vectorized_Features_K30_Generalization\pasto\pasto_57_features.csv'
]

original_files = [
    r'C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Preprocessing_10_Generalization\adoquin\adoquin_test_3_0.csv',
    r'C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Preprocessing_10_Generalization\asfalto\asfalto_0.csv',
    r'C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Preprocessing_10_Generalization\concreto\cemento_ccu_47.csv',
    r'C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Preprocessing_10_Generalization\grava\grava_30.csv',
    r'C:\Users\USER\DESKTOP\PEF\Terrain-Traversability-Analysis\Preprocessing_10_Generalization\pasto\pasto_57.csv'
]

color_dict = {
    'Cobblestone': '#FF4500', # OrangeRed
    'Asphalt': '#00008B',     # DarkBlue 
    'Concrete': '#00BFFF',    # DeepSkyBlue 
    'Gravel': '#FFD700',      # Gold 
    'Grass': '#32CD32'        # LimeGreen
}

class_names = ['Cobblestone', 'Asphalt', 'Concrete', 'Gravel', 'Grass']

for file, original_file, class_name in zip(files, original_files, class_names):
    print(f"\n=========================================")
    print(f" Plotting Class: {class_name.title()}")
    print(f"=========================================")
    
    df = pl.read_csv(file)
    predictions = model.predict(df)

    original_file_df = pl.read_csv(original_file)

    original_file_df = original_file_df.with_columns([
        pl.Series("terrain_label", [terrain_map[p] for p in predictions])
    ])

    plot_df = original_file_df.to_pandas()
    
    plot_df = plot_df.sort_values('terrain_label')

    fig = px.scatter_3d(
        plot_df, 
        x='X',   
        y='Y',
        z='Z',
        color='terrain_label', 
        color_discrete_map=color_dict,
        template='plotly_white',
        hover_data={'X': ':.2f', 'Y': ':.2f', 'Z': ':.2f'} 
    )
    
    fig.update_traces(marker=dict(size=1.5, opacity=1.0)) 
        
    fig.update_layout(
        title=dict(
            text=f"Semantic Segmentation - Ground Truth: {class_name.title()}",
            y=0.98, 
            x=0.5,
            xanchor='center',
            yanchor='top'
        ),
        scene=dict(
            aspectmode='data', # Mantiene la proporción real de los metros (1m en X = 1m en Y)
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)',
            
            # --- LIMITS X AND Y ---
            xaxis=dict(
                range=[-3, 3], # Fuerza la vista desde -3 a 3
                showbackground=False, showgrid=True, gridcolor='darkgrey', gridwidth=1, zerolinecolor='darkgrey'
            ), 
            yaxis=dict(
                range=[0, 3],  # Fuerza la vista desde 0 a 3
                showbackground=False, showgrid=True, gridcolor='darkgrey', gridwidth=1, zerolinecolor='darkgrey'
            ),
            zaxis=dict(
                showbackground=False, showgrid=True, gridcolor='darkgrey', gridwidth=1, zerolinecolor='darkgrey'
            ),
            bgcolor='white',
            
            # --- 2. BEV (Bird's-Eye View) ---
            camera=dict(
                up=dict(x=0, y=1, z=0),   
                center=dict(x=0, y=0, z=0), 
                # Aumenté Z de 2.5 a 3.5 para alejar la cámara y que quepan los 6 metros de X
                eye=dict(x=0, y=0, z=3.5) 
            )
        ), 
        margin=dict(l=10, r=10, b=10, t=60),
        legend=dict(
            title_text='Terrain Class',
            itemsizing='constant',
            bgcolor='rgba(255,255,255,0.8)', 
            bordercolor='darkgrey',
            borderwidth=1,
            yanchor="top",
            y=0.95,
            xanchor="left",
            x=0.85
        )
    )
    
    fig.show()
    
    # Calculate percentages and totals with schema fix
    summary_df = original_file_df.group_by("terrain_label").count().with_columns(
        (pl.col("count") / original_file_df.height * 100).round(2).alias("percentage_%")
    ).sort("count", descending=True)
    
    total_row = pl.DataFrame({
        "terrain_label": ["TOTAL"],
        "count": [original_file_df.height],
        "percentage_%": [100.00]
    }).cast(summary_df.schema) 
    
    print(pl.concat([summary_df, total_row]))


 Plotting Class: Cobblestone


shape: (5, 3)
┌───────────────┬───────┬──────────────┐
│ terrain_label ┆ count ┆ percentage_% │
│ ---           ┆ ---   ┆ ---          │
│ str           ┆ u32   ┆ f64          │
╞═══════════════╪═══════╪══════════════╡
│ Concrete      ┆ 9365  ┆ 41.64        │
│ Asphalt       ┆ 9270  ┆ 41.21        │
│ Cobblestone   ┆ 3850  ┆ 17.12        │
│ Gravel        ┆ 7     ┆ 0.03         │
│ TOTAL         ┆ 22492 ┆ 100.0        │
└───────────────┴───────┴──────────────┘

 Plotting Class: Asphalt


shape: (2, 3)
┌───────────────┬───────┬──────────────┐
│ terrain_label ┆ count ┆ percentage_% │
│ ---           ┆ ---   ┆ ---          │
│ str           ┆ u32   ┆ f64          │
╞═══════════════╪═══════╪══════════════╡
│ Asphalt       ┆ 24020 ┆ 100.0        │
│ TOTAL         ┆ 24020 ┆ 100.0        │
└───────────────┴───────┴──────────────┘

 Plotting Class: Concrete


shape: (5, 3)
┌───────────────┬───────┬──────────────┐
│ terrain_label ┆ count ┆ percentage_% │
│ ---           ┆ ---   ┆ ---          │
│ str           ┆ u32   ┆ f64          │
╞═══════════════╪═══════╪══════════════╡
│ Concrete      ┆ 21321 ┆ 98.33        │
│ Cobblestone   ┆ 193   ┆ 0.89         │
│ Asphalt       ┆ 104   ┆ 0.48         │
│ Gravel        ┆ 66    ┆ 0.3          │
│ TOTAL         ┆ 21684 ┆ 100.0        │
└───────────────┴───────┴──────────────┘

 Plotting Class: Gravel


shape: (6, 3)
┌───────────────┬───────┬──────────────┐
│ terrain_label ┆ count ┆ percentage_% │
│ ---           ┆ ---   ┆ ---          │
│ str           ┆ u32   ┆ f64          │
╞═══════════════╪═══════╪══════════════╡
│ Gravel        ┆ 12129 ┆ 50.33        │
│ Cobblestone   ┆ 8349  ┆ 34.64        │
│ Grass         ┆ 2002  ┆ 8.31         │
│ Concrete      ┆ 1511  ┆ 6.27         │
│ Asphalt       ┆ 108   ┆ 0.45         │
│ TOTAL         ┆ 24099 ┆ 100.0        │
└───────────────┴───────┴──────────────┘

 Plotting Class: Grass


shape: (5, 3)
┌───────────────┬───────┬──────────────┐
│ terrain_label ┆ count ┆ percentage_% │
│ ---           ┆ ---   ┆ ---          │
│ str           ┆ u32   ┆ f64          │
╞═══════════════╪═══════╪══════════════╡
│ Grass         ┆ 22235 ┆ 94.32        │
│ Gravel        ┆ 1152  ┆ 4.89         │
│ Concrete      ┆ 186   ┆ 0.79         │
│ Cobblestone   ┆ 2     ┆ 0.01         │
│ TOTAL         ┆ 23575 ┆ 100.0        │
└───────────────┴───────┴──────────────┘
